<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/01_exploracion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Exploración de los datos

Qué hay, cuánto de cada cosa, cómo son las imágenes y las cajas. Anotar lo que llame la atención al final.

In [ ]:
# --- Setup: correr siempre primero ---
INTEGRANTE = "nombre"  # tu nombre: separa tus resultados de los del resto del equipo

import os, sys
if "google.colab" in sys.modules:
    REPO = "/content/PDI-TP"
    if os.path.isdir(REPO):
        !git -C {REPO} pull -q
    else:
        !git clone -q https://github.com/candeperles/PDI-TP.git {REPO}
    %cd {REPO}
else:
    REPO = os.path.abspath("..")  # corriendo en local desde notebooks/
sys.path.insert(0, REPO)

from pdi_tp import entorno, datos, resultados
rutas = entorno.preparar(INTEGRANTE)

## Imágenes y etiquetas

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image

imagenes = datos.listar_imagenes(rutas)
filas = []
for img in imagenes:
    cajas = datos.leer_cajas(datos.label_de(rutas, img))
    with Image.open(img) as im:
        tamano, modo = im.size, im.mode
    for clase, xc, yc, w, h in (cajas if len(cajas) else [[np.nan] * 5]):
        filas.append({"imagen": img.name, "tamano": tamano, "modo": modo,
                      "clase": clase, "xc": xc, "yc": yc, "w": w, "h": h})
cajas = pd.DataFrame(filas)

por_imagen = cajas.groupby("imagen")["clase"].count()
print(f"Imágenes: {len(imagenes)}")
print(f"Con paneles: {(por_imagen > 0).sum()}   sin paneles: {(por_imagen == 0).sum()}")
una_por_imagen = cajas.drop_duplicates("imagen")
print(f"Tamaños: {una_por_imagen['tamano'].value_counts().to_dict()}   modos: {una_por_imagen['modo'].value_counts().to_dict()}")
print("Cajas por clase:")
print(cajas["clase"].value_counts().sort_index())

## Cajas por imagen y tamaño de las cajas

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
por_imagen.plot.hist(bins=30, ax=axs[0], title="Cajas por imagen")
(cajas["w"] * 256).plot.hist(bins=40, ax=axs[1], title="Ancho de caja (px)")
(cajas["h"] * 256).plot.hist(bins=40, ax=axs[2], title="Alto de caja (px)")
plt.tight_layout()

## Ejemplos con cajas dibujadas

In [ ]:
import random

random.seed(0)
muestra = random.sample(imagenes, 12)
fig, axs = plt.subplots(3, 4, figsize=(16, 12))
for ax, img in zip(axs.flat, muestra):
    datos.dibujar_cajas(ax, rutas, img)
    ax.set_title(img.name[:30], fontsize=8)
plt.tight_layout()

## Observaciones

- ...